<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Classify clothing images

**[Classify clothing images](https://learning.nextia-ai.com/courses/deep-learning/m08/classify-clothing-images/)** · Deep Learning and Transformers Explained · Module 8, case study 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** build an image classifier end to end on real product photos: read the raw files into tensors, set a baseline, train a multilayer network and a small convolutional network with early stopping, read the confusion matrix and the errors, test a data augmentation, choose and test the final model once, save it with an experiment record, and use it on new images.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 2 minutes on the course's test computer (the networks train on 10,000 of the 60,000 training images, to keep it short on a CPU); allow up to 10 minutes on a free Colab CPU. |
| **Needs** | An internet connection for the setup cells (31 MB of data). No account. PyTorch, scikit-learn and matplotlib (preinstalled in Colab and Kaggle). |
| **You should know** | Modules 1 to 4 of this course, most of all [Tensors and shapes](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/), [Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/) and [Overfitting controls](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/). |
| **Data** | Fashion-MNIST (Zalando Research, 2017), MIT License. 70,000 real product photos, 28 × 28 grey pixels, ten categories. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/classify-clothing-images/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M08-L01-classify-clothing-images/classify-clothing-images-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: scikit-learn, matplotlib and the images

Run the next two cells. The first one installs scikit-learn and matplotlib if they are missing (Colab and Kaggle have them). The second one makes a folder `clothing-images` with a `data` folder in it, downloads the four Fashion-MNIST files (31 MB in total) from Zalando Research's repository on GitHub, and checks each file's checksum (SHA-256). If GitHub does not answer, it tries the dataset's second address. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready:` and the four files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `clothing-images` (in Colab: the **Files** panel on the left), then run the cell again.

Data: Fashion-MNIST by Zalando Research (Xiao, Rasul and Vollgraf, 2017), https://github.com/zalandoresearch/fashion-mnist, MIT License, Copyright © 2017 Zalando SE. The images are real product photos from Zalando's catalogue, shrunk to 28 × 28 grey pixels. The shop in this case study, Brightloom, is made up.

In [ ]:
try:
    import sklearn, matplotlib
except ImportError:  # a local environment without them
    %pip install -q --disable-pip-version-check scikit-learn matplotlib
    import sklearn, matplotlib
print("scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

In [ ]:
# Setup: download the four Fashion-MNIST files into clothing-images/data/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

SOURCES = ["https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/",
           "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/"]
FILES = {
    "train-images-idx3-ubyte.gz": "3aede38d61863908ad78613f6a32ed271626dd12800ba2636569512369268a84",
    "train-labels-idx1-ubyte.gz": "a04f17134ac03560a47e3764e11b92fc97de4d1bfaf8ba1a3aa29af54cc90845",
    "t10k-images-idx3-ubyte.gz": "346e55b948d973a97e58d2351dde16a484bd415d4595297633bb08f03db6a073",
    "t10k-labels-idx1-ubyte.gz": "67da17c76eaffca5446c3361aaab5c3cd6d1c2608764d35dfb1850b086bf8dd5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("clothing-images").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(name, path):
    for source in SOURCES:
        try:
            urllib.request.urlretrieve(source + name, path)
            return
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(path), source + name]).returncode == 0:
                return
    raise SystemExit(f"Could not download {name}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder clothing-images and run again.")
os.chdir(WORK)  # work in this folder: the model and the experiment record are saved here
def size(path):
    n = path.stat().st_size
    return f"{n / 1e6:.1f} MB" if n >= 1e6 else f"{n / 1e3:.0f} KB"

print("Ready:", ", ".join(f"data/{n} ({size(Path('data', n))})" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: two drawing helpers

`show_images()` draws images in a grid with a title over each, and `plot_curves()` draws the training and validation loss per epoch. Run the cell. It prints nothing.

In [ ]:
import matplotlib.pyplot as plt

def show_images(imgs, titles, columns=8, size=1.4):
    """Draw grey images (28 x 28 arrays or tensors) in a grid, with a title over each."""
    rows = (len(imgs) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(columns * size, rows * (size + 0.35)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img, title in zip(axes.flat, imgs, titles):
        ax.imshow(np.asarray(img).reshape(28, 28), cmap="gray", vmin=0, vmax=np.asarray(img).max() or 1)
        ax.set_title(title, fontsize=8)
    plt.tight_layout()
    plt.show()

def plot_curves(runs):
    """Training loss (dashed) and validation loss (solid) per epoch, for {name: history}."""
    for name, history in runs.items():
        epochs = [row["epoch"] for row in history]
        line, = plt.plot(epochs, [row["valid_loss"] for row in history], marker="o", ms=3, label=f"{name}, validation")
        plt.plot(epochs, [row["train_loss"] for row in history], ls="--", color=line.get_color(), label=f"{name}, training")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Brightloom is an online clothing marketplace (made up for this case study). Sellers upload a product photo and choose one of ten categories. Many choose the wrong one, so the product appears in the wrong aisle and customers do not find it. Ines, the catalogue lead, wants the category **suggested from the photo**: filled in when the model is sure, and two choices offered when it is not. Kwame, the data scientist, builds the model. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/classify-clothing-images/) tells the full story; this notebook runs every step.

## 2. The data

The files are in the IDX format: a short header, then one byte per pixel. `read_images()` reads the three numbers of the header (count, rows, columns) and the pixels after it. Run the cell.

In [ ]:
# [load]
import gzip
import numpy as np

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


def read_images(path):
    """IDX images: 16 header bytes (a file code, count, rows, columns), then one byte per pixel."""
    raw = gzip.open(path).read()
    count, rows, cols = (int.from_bytes(raw[i:i + 4], "big") for i in (4, 8, 12))
    return np.frombuffer(raw, dtype=np.uint8, offset=16).reshape(count, rows, cols)


def read_labels(path):
    """IDX labels: 8 header bytes (a file code, count), then one byte per label."""
    return np.frombuffer(gzip.open(path).read(), dtype=np.uint8, offset=8)


images = read_images("data/train-images-idx3-ubyte.gz")
labels = read_labels("data/train-labels-idx1-ubyte.gz")
test_images = read_images("data/t10k-images-idx3-ubyte.gz")
test_labels = read_labels("data/t10k-labels-idx1-ubyte.gz")
print(images.shape, images.dtype, labels.shape)
print(test_images.shape, test_labels.shape)
print("first image's label:", labels[0], CLASSES[labels[0]])

> **Check.** You should see:
> ```text
> (60000, 28, 28) uint8 (60000,)
> (10000, 28, 28) (10000,)
> first image's label: 9 Ankle boot
> ```

**What this tells us.** The training file has 60,000 images of 28 rows × 28 columns, one byte (0 to 255) per pixel, and one label per image. The test file has 10,000. Each label is a number from 0 to 9; `CLASSES` gives its name.

Now the first checks: the pixel range, the images per class, and whether any test image is also in the training file (a copy would make the test too easy).

In [ ]:
# [checks]
print("pixel values from", images.min(), "to", images.max())
print("training images per class:", np.bincount(labels).tolist())
print("test images per class:    ", np.bincount(test_labels).tolist())
train_keys = {img.tobytes() for img in images}
print("repeated images in the training file:", len(images) - len(train_keys))
print("test images also in the training file:", sum(img.tobytes() in train_keys for img in test_images))

> **Check.** You should see:
> ```text
> pixel values from 0 to 255
> training images per class: [6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]
> test images per class:     [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
> repeated images in the training file: 0
> test images also in the training file: 0
> ```

**What this tells us.** Every class has exactly 6,000 training and 1,000 test images, so the classes are balanced: guessing one class is right 10% of the time, and accuracy is a fair measure. No image is repeated, and no test image is a copy of a training image.

Look at some images: the first two of each class.

In [ ]:
picks = [i for k in range(10) for i in np.where(labels == k)[0][:2]]
show_images(images[picks], [CLASSES[labels[i]] for i in picks], columns=10)

And the numbers behind one image. The next cell prints an 8 × 8 corner of the first image (an ankle boot), where the toe begins.

> **Predict.** What number do you expect for the black background?

In [ ]:
# [pixels]
print(CLASSES[labels[0]])
print(images[0, 16:24, 0:8])

> **Check.** You should see:
> ```text
> Ankle boot
> [[  0   0   0   0  18  44  82 107]
>  [  0  57 187 208 224 221 224 208]
>  [  3 202 228 224 221 211 211 214]
>  [ 98 233 198 210 222 229 229 234]
>  [ 75 204 212 204 193 205 211 225]
>  [ 48 203 183 194 213 197 185 190]
>  [  0 122 219 193 179 171 183 196]
>  [  0   0  74 189 212 191 175 172]]
> ```

**What this tells us.** The background is 0 (black) and the boot is bright (about 200). The model gets only these numbers: 784 of them per image.

## 3. Split, and images as tensors

The test file stays sealed until section 9. From the training file, take a stratified sample of 10,000 images to train on and 5,000 others to validate on. Ten thousand is enough to show every idea, and each network trains in about a minute on a CPU. Then turn them into tensors of shape (images, channels, rows, columns), as PyTorch's image layers expect. A grey image has **one channel**; a colour photo would have three (red, green, blue). Dividing by 255 puts every pixel between 0 and 1.

In [ ]:
# [split]
import torch
from sklearn.model_selection import train_test_split

rows = np.arange(len(labels))
train_rows, rest = train_test_split(rows, train_size=10_000, stratify=labels, random_state=0)
valid_rows, _ = train_test_split(rest, train_size=5_000, stratify=labels[rest], random_state=0)


def to_tensors(imgs, labs):
    """uint8 images (N, 28, 28) -> floats (N, 1, 28, 28) from 0 to 1, and int64 labels (N,)."""
    x = torch.tensor(imgs, dtype=torch.float32).unsqueeze(1) / 255
    return x, torch.tensor(labs, dtype=torch.long)


x_train, y_train = to_tensors(images[train_rows], labels[train_rows])
x_valid, y_valid = to_tensors(images[valid_rows], labels[valid_rows])
x_test, y_test = to_tensors(test_images, test_labels)
show_shape("x_train", x_train)
show_shape("x_valid", x_valid)
show_shape("x_test", x_test)
print("validation images per class:", torch.bincount(y_valid).tolist())

> **Check.** You should see:
> ```text
> x_train      shape (10000, 1, 28, 28)
> x_valid      shape (5000, 1, 28, 28)
> x_test       shape (10000, 1, 28, 28)
> validation images per class: [500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
> ```

`unsqueeze(1)` adds the channel dimension at position 1: (10000, 28, 28) becomes (10000, 1, 28, 28).

> **Your turn.** What share of all pixels in `x_train` is exactly 0 (black background)? Put it in `background_share` as a Python float, then run the check cell.

In [ ]:
# [task1]
background_share = ...  # the share of pixels in x_train that are exactly 0

In [ ]:
expect('background_share', background_share, 0.5024, tolerance=0.001)

## 4. Baselines

Two baselines. The majority class is right 10% of the time, because the classes are balanced. Logistic regression, from the machine learning course, reads the 784 pixels as 784 features: `flatten(1)` keeps the first dimension (the images) and lays out everything else in one row, (10000, 784). It takes a few seconds.

In [ ]:
# [logreg]
from sklearn.linear_model import LogisticRegression

majority = int(torch.bincount(y_train).argmax())
print("majority class, validation accuracy:", round((y_valid == majority).float().mean().item(), 4))
logreg = LogisticRegression(max_iter=1000)
logreg.fit(x_train.flatten(1).numpy(), y_train.numpy())       # 784 pixel features per image
print("logistic regression, validation accuracy:", round(logreg.score(x_valid.flatten(1).numpy(), y_valid.numpy()), 4))

> **Check.** You should see:
> ```text
> majority class, validation accuracy: 0.1
> logistic regression, validation accuracy: 0.8346
> ```

**What this tells us.** A linear model on raw pixels already gets about 5 images in 6 right. That is the number to beat: a network must do clearly better to be worth its cost.

## 5. Training with early stopping

`fit()` is the training loop of [Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/): shuffled batches of 64, the five steps, and an evaluation pass after every epoch. It adds **early stopping** from [Overfitting controls](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/): it keeps a copy of the weights with the best validation loss, stops after 5 epochs without a new best, and puts the best weights back. The `flip` lines are for section 8; leave them for now. Run the cell. It prints nothing.

In [ ]:
# [fit]
from torch import nn


def set_seed(seed=0):
    torch.manual_seed(seed)


def count(model):
    return sum(p.numel() for p in model.parameters())


def evaluate(model, x, y, batch_size=500):
    """Mean loss and accuracy, in evaluation mode and without gradients."""
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total, correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            scores = model(x[i:i + batch_size])
            total += loss_fn(scores, y[i:i + batch_size]).item()
            correct += (scores.argmax(1) == y[i:i + batch_size]).sum().item()
    return total / len(x), correct / len(x)


def fit(model, x_train, y_train, x_valid, y_valid, epochs=30, patience=5, flip=False, seed=0, batch_size=64, lr=1e-3):
    """Adam, shuffled mini-batches, and early stopping on the validation loss. Returns (history, best row)."""
    set_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history, best, best_state, waited = [], None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(x_train))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            images, labels = x_train[batch], y_train[batch]
            if flip:  # data augmentation: mirror about half of the batch
                mirror = torch.rand(len(batch)) < 0.5
                images = torch.where(mirror[:, None, None, None], images.flip(-1), images)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, x_train, y_train)
        valid_loss, valid_acc = evaluate(model, x_valid, y_valid)
        row = {"epoch": epoch, "train_loss": round(train_loss, 4), "train_acc": round(train_acc, 4),
               "valid_loss": round(valid_loss, 4), "valid_acc": round(valid_acc, 4)}
        history.append(row)
        if best is None or valid_loss < best["valid_loss"]:   # a new best: keep a copy of the weights
            best, waited = row, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:                             # no new best for `patience` epochs: stop
                break
    model.load_state_dict(best_state)                          # use the best epoch's weights
    return history, best

## 6. A multilayer network

The network of this course's earlier modules, with pixels in place of word counts: `nn.Flatten()` makes each image a row of 784 numbers, then one hidden layer of 128 ReLU neurons and 10 scores. It takes about 5 seconds.

In [ ]:
# [mlp]
class MLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.layers = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, 10))

    def forward(self, x):
        return self.layers(x)


set_seed(0)
mlp = MLP()
print("weights:", count(mlp))
mlp_history, mlp_best = fit(mlp, x_train, y_train, x_valid, y_valid)
print("stopped after epoch", len(mlp_history), "| best:", mlp_best)

> **Check.** You should see:
> ```text
> weights: 101770
> stopped after epoch 24 | best: {'epoch': 19, 'train_loss': 0.2207, 'train_acc': 0.9249, 'valid_loss': 0.3723, 'valid_acc': 0.8712}
> ```

**What this tells us.** The network beats logistic regression by a few points. Early stopping kept the best epoch and stopped 5 epochs later.

### What the flat network ignores

`nn.Flatten()` throws away which pixels are neighbours. Test it: put the 784 pixels of every image in one fixed random order, so the images look like noise to a person, and train the same network again.

> **Predict.** Will the accuracy fall a lot, a little, or not at all?

In [ ]:
# [scramble]
perm = torch.randperm(784, generator=torch.Generator().manual_seed(0))  # one fixed new order


def scramble(x):
    return x.flatten(1)[:, perm].reshape(-1, 1, 28, 28)


show_images([x_train[0], scramble(x_train[:1])[0]], ["as you see it", "scrambled"], columns=2, size=2)
set_seed(0)
_, scrambled_best = fit(MLP(), scramble(x_train), y_train, scramble(x_valid), y_valid)
print("scrambled pixels, best:", scrambled_best)

> **Check.** You should see:
> ```text
> scrambled pixels, best: {'epoch': 19, 'train_loss': 0.225, 'train_acc': 0.9209, 'valid_loss': 0.3661, 'valid_acc': 0.8708}
> ```

**What this tells us.** The accuracy is almost the same. The flat network never knew where a pixel was: a picture and its scrambled copy are the same problem to it. So it must learn "a collar here" separately for every position, from examples. The next test shows the cost: move every validation image one, two or three pixels to the right.

In [ ]:
# [shift_mlp]
for pixels in (1, 2, 3):
    moved = torch.roll(x_valid, shifts=pixels, dims=3)   # the background is black, so this moves the item
    print(f"moved {pixels} pixel(s) right: MLP accuracy {evaluate(mlp, moved, y_valid)[1]:.4f}")

> **Check.** You should see:
> ```text
> moved 1 pixel(s) right: MLP accuracy 0.7602
> moved 2 pixel(s) right: MLP accuracy 0.4308
> moved 3 pixel(s) right: MLP accuracy 0.2330
> ```

**What this tells us.** A shirt moved by two pixels is still a shirt, but the network's accuracy falls by about half. Its weights belong to positions, not to shapes.

## 7. A small convolutional network

A **convolution** slides a small **filter**, a 3 × 3 grid of weights, over the image. At each position it multiplies the 9 pixels under it by the 9 weights and adds them up: one number per position. The same 9 weights are used everywhere, so a pattern (an edge, a corner) is found wherever it is. The network learns the weights of its filters, as it learns any other weights. The lesson page shows one step in a picture.

> **Your turn.** Here is a 3 × 3 window of pixels from the edge of a trouser leg, and a filter that responds to dark-to-bright edges from left to right. Compute the filter's answer for this window: multiply element by element and add. Put it in `response`, a Python float.

In [ ]:
# [task2]
window = torch.tensor([[0.00, 0.02, 0.98],
                       [0.00, 0.00, 0.92],
                       [0.00, 0.47, 0.88]])
edge_filter = torch.tensor([[-1.0, 0.0, 1.0],
                            [-1.0, 0.0, 1.0],
                            [-1.0, 0.0, 1.0]])
response = ...  # multiply element by element, then add up

In [ ]:
expect('response', response, 2.78, tolerance=0.001)

The network below has two convolution layers (16 and then 32 filters of 3 × 3). After each one, ReLU, and **max pooling**, which keeps the largest value of each 2 × 2 square: the image becomes half as wide and half as high, and a small shift matters less. Then the course's familiar layers make 10 scores. Run the cell: it prints the shape after every layer, for a batch of 64 images.

In [ ]:
# [cnn_shapes]
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 filters on 1 channel
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 filters on 16 channels
        )
        self.classify = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.classify(self.features(x))


set_seed(0)
cnn = SmallCNN()
x = x_train[:64]
show_shape("input", x)
for layer in cnn.features:
    x = layer(x)
    show_shape(type(layer).__name__, x)
show_shape("scores", cnn(x_train[:64]))
print("weights:", count(cnn), "| first convolution:", count(cnn.features[0]))

> **Check.** You should see:
> ```text
> input        shape (64, 1, 28, 28)
> Conv2d       shape (64, 16, 28, 28)
> ReLU         shape (64, 16, 28, 28)
> MaxPool2d    shape (64, 16, 14, 14)
> Conv2d       shape (64, 32, 14, 14)
> ReLU         shape (64, 32, 14, 14)
> MaxPool2d    shape (64, 32, 7, 7)
> scores       shape (64, 10)
> weights: 105866 | first convolution: 160
> ```

`padding=1` adds a border of zeros, so a 3 × 3 window fits at every pixel and the size stays 28 × 28. The first convolution has 16 filters × 9 weights + 16 biases = 160 weights. The whole network has about as many weights as the MLP; the difference is how they are used.

Now train it. It takes about 40 seconds on the course's test computer, perhaps two minutes on a free Colab CPU.

In [ ]:
# [cnn]
cnn_history, cnn_best = fit(cnn, x_train, y_train, x_valid, y_valid)
print("stopped after epoch", len(cnn_history), "| best:", cnn_best)
plot_curves({"MLP": mlp_history, "CNN": cnn_history})

> **Check.** You should see:
> ```text
> stopped after epoch 24 | best: {'epoch': 19, 'train_loss': 0.1723, 'train_acc': 0.9396, 'valid_loss': 0.3221, 'valid_acc': 0.8912}
> ```

**What this tells us.** The convolutional network beats the MLP by about two points of validation accuracy and has a lower validation loss. In the chart, each network's training loss keeps falling while its validation loss flattens and then rises a little: early stopping chose the epoch before that.

Is it less fragile when the image moves? The same test as before:

In [ ]:
# [shift_cnn]
for pixels in (1, 2, 3):
    moved = torch.roll(x_valid, shifts=pixels, dims=3)
    print(f"moved {pixels} pixel(s) right: MLP {evaluate(mlp, moved, y_valid)[1]:.4f}   CNN {evaluate(cnn, moved, y_valid)[1]:.4f}")

> **Check.** You should see:
> ```text
> moved 1 pixel(s) right: MLP 0.7602   CNN 0.8572
> moved 2 pixel(s) right: MLP 0.4308   CNN 0.7346
> moved 3 pixel(s) right: MLP 0.2330   CNN 0.4678
> ```

**What this tells us.** The convolutional network loses much less, but it is not immune: at three pixels it also fails often. It never saw a moved item in training, because every Fashion-MNIST photo is centred.

### A fair comparison

Every model so far, on the same 5,000 validation images, after the same split:

In [ ]:
# [compare]
import pandas as pd

comparison = pd.DataFrame([
    {"model": "majority class", "weights": 0, "valid_acc": round((y_valid == majority).float().mean().item(), 4), "valid_loss": None},
    {"model": "logistic regression", "weights": logreg.coef_.size + logreg.intercept_.size,
     "valid_acc": logreg.score(x_valid.flatten(1).numpy(), y_valid.numpy()), "valid_loss": None},
    {"model": "MLP", "weights": count(mlp), "valid_acc": mlp_best["valid_acc"], "valid_loss": mlp_best["valid_loss"]},
    {"model": "small CNN", "weights": count(cnn), "valid_acc": cnn_best["valid_acc"], "valid_loss": cnn_best["valid_loss"]},
])
print(comparison.round(4).to_string(index=False))

> **Check.** You should see:
> ```text
>               model  weights  valid_acc  valid_loss
>      majority class        0     0.1000         NaN
> logistic regression     7850     0.8346         NaN
>                 MLP   101770     0.8712      0.3723
>           small CNN   105866     0.8912      0.3221
> ```

## 8. Where it fails, and a flip

The confusion matrix of [Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/#the-confusion-matrix) in the machine learning course, with ten classes: rows are the true class, columns the predicted class.

In [ ]:
# [confusion]
from sklearn.metrics import confusion_matrix


def predict(model, x):
    """Probabilities for every class, in evaluation mode, in batches of 500."""
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x[i:i + 500]).softmax(1) for i in range(0, len(x), 500)])


valid_probs = predict(cnn, x_valid)
valid_pred = valid_probs.argmax(1)
cm = confusion_matrix(y_valid, valid_pred)
print(pd.DataFrame(cm, index=CLASSES, columns=[c[:7] for c in CLASSES]).to_string())

> **Check.** You should see:
> ```text
>              T-shirt  Trouser  Pullove  Dress  Coat  Sandal  Shirt  Sneaker  Bag  Ankle b
> T-shirt/top      421        1       10     20     1       0     46        0    1        0
> Trouser            1      493        0      5     1       0      0        0    0        0
> Pullover           7        2      435      3    30       0     23        0    0        0
> Dress              7        6        4    452    19       0     11        0    1        0
> Coat               0        2       41     11   418       0     28        0    0        0
> Sandal             0        0        0      0     0     474      0       16    1        9
> Shirt             51        1       50     17    48       0    330        0    2        1
> Sneaker            0        0        0      0     0       9      0      476    0       15
> Bag                5        0        2      1     4       4      6        0  477        1
> Ankle boot         0        1        0      0     0       0      0       19    0      480
> ```

> **Your turn.** What share of the true shirts does the network call "Shirt"? That is the recall of the class Shirt (row 6). Put it in `shirt_recall` as a Python float.

In [ ]:
# [task3]
shirt_recall = ...  # cm[6, 6] divided by the number of true shirts

In [ ]:
expect('shirt_recall', shirt_recall, 0.66, tolerance=0.001)

Now look at the errors themselves: the 10 that the network is most sure of.

In [ ]:
# [errors]
wrong = torch.nonzero(valid_pred != y_valid).flatten()
sure_first = wrong[valid_probs[wrong, valid_pred[wrong]].argsort(descending=True)]
print("wrong:", len(wrong), "of", len(y_valid))
for i in sure_first[:10].tolist():
    print(f"label {CLASSES[y_valid[i]]:<12} model {CLASSES[valid_pred[i]]:<12} probability {valid_probs[i, valid_pred[i]]:.3f}")
show_images(x_valid[sure_first[:10]], [f"{CLASSES[y_valid[i]]}\nmodel: {CLASSES[valid_pred[i]]}" for i in sure_first[:10].tolist()], columns=5, size=1.8)

> **Check.** You should see:
> ```text
> wrong: 544 of 5000
> label Ankle boot   model Sneaker      probability 1.000
> label T-shirt/top  model Dress        probability 1.000
> label Sneaker      model Ankle boot   probability 0.999
> label Ankle boot   model Sneaker      probability 0.999
> label Sandal       model Sneaker      probability 0.998
> label Shirt        model Trouser      probability 0.998
> label Bag          model Shirt        probability 0.998
> label Sandal       model Sneaker      probability 0.998
> label Sneaker      model Ankle boot   probability 0.998
> label Sneaker      model Ankle boot   probability 0.997
> ```

Look at the pictures before you blame the model. Some of them would puzzle a person too at 28 × 28 pixels, and some labels look wrong.

### Data augmentation: a flip

A shirt flipped from left to right is still a shirt, so a flipped copy is a free extra training image with a true label. `fit()` already has the two lines: in each batch, `torch.rand(len(batch)) < 0.5` chooses about half of the images, and `images.flip(-1)` mirrors them along the last dimension, the columns.

> **Your turn.** Flip the first training image, `x_train[0]` (shape (1, 28, 28)), from left to right, and put it in `flipped`.

In [ ]:
# [task4]
flipped = ...  # x_train[0] mirrored from left to right

In [ ]:
mirrored = isinstance(flipped, torch.Tensor) and flipped.shape == (1, 28, 28) and torch.equal(flipped[0, :, 0], x_train[0, 0, :, -1])
expect("flipped is x_train[0] mirrored from left to right", "yes" if mirrored else "no", "yes")

Train the same network with the flip, from the same seed. It takes about 40 seconds.

> **Predict.** Will the validation accuracy go up by more than one point?

In [ ]:
# [flip]
set_seed(0)
cnn_flip = SmallCNN()
flip_history, flip_best = fit(cnn_flip, x_train, y_train, x_valid, y_valid, flip=True)
print("without flip:", cnn_best)
print("with flip:   ", flip_best)

> **Check.** You should see:
> ```text
> without flip: {'epoch': 19, 'train_loss': 0.1723, 'train_acc': 0.9396, 'valid_loss': 0.3221, 'valid_acc': 0.8912}
> with flip:    {'epoch': 16, 'train_loss': 0.2482, 'train_acc': 0.9103, 'valid_loss': 0.3206, 'valid_acc': 0.8854}
> ```

**What this tells us.** The flip changes the result by less than one point, and not clearly in one direction: one run cannot tell. The lesson page shows five seeds of each and why the flip barely helps here (every shoe in the file faces the same way). Kwame's rule, written before he looked: keep the flip only if it beats the plain network by more than the spread between seeds. It does not, so the final model is the plain convolutional network, `cnn`.

## 9. The operating point and the final test

Ines wants the category filled in only when the model is sure. On the validation images, look at how many images pass a probability threshold, and how often the model is right on those.

In [ ]:
# [threshold]
top = valid_probs.max(1).values
for threshold in (0.5, 0.8, 0.9, 0.95):
    sure = top >= threshold
    in_top2 = (valid_probs[~sure].topk(2, dim=1).indices == y_valid[~sure, None]).any(1)
    print(f"threshold {threshold}: filled in {sure.float().mean():.3f}, right there {(valid_pred[sure] == y_valid[sure]).float().mean():.4f}, "
          f"others: true class in the top 2 {in_top2.float().mean():.4f}")

> **Check.** You should see:
> ```text
> threshold 0.5: filled in 0.976, right there 0.9035, others: true class in the top 2 0.7227
> threshold 0.8: filled in 0.819, right there 0.9556, others: true class in the top 2 0.8826
> threshold 0.9: filled in 0.735, right there 0.9763, others: true class in the top 2 0.9011
> threshold 0.95: filled in 0.662, right there 0.9861, others: true class in the top 2 0.9201
> ```

> **Your turn.** At a threshold of 0.7, what share of validation images is filled in? Put it in `share_at_07` as a Python float.

In [ ]:
# [task5]
share_at_07 = ...  # the share of validation images whose top probability is at least 0.7

In [ ]:
expect('share_at_07', share_at_07, 0.874, tolerance=0.001)

Kwame and Ines choose 0.9: about three images in four are filled in, and about 98 in 100 of those are right. For the others, the seller sees the two most likely categories.

Every choice is now made: the plain convolutional network, the best epoch, the threshold 0.9. Open the test set once.

> **Warning.** Run the next cell once. Do not change the model or the threshold after you read it.

In [ ]:
# [test]
test_probs = predict(cnn, x_test)
test_pred = test_probs.argmax(1)
sure = test_probs.max(1).values >= 0.9
in_top2 = (test_probs[~sure].topk(2, dim=1).indices == y_test[~sure, None]).any(1)
print("test accuracy:", round((test_pred == y_test).float().mean().item(), 4))
print(f"filled in {sure.float().mean():.3f}, right there {(test_pred[sure] == y_test[sure]).float().mean():.4f}, "
      f"others: true class in the top 2 {in_top2.float().mean():.4f}")
test_cm = confusion_matrix(y_test, test_pred)
print("recall per class:", dict(zip(CLASSES, (test_cm.diagonal() / test_cm.sum(1)).round(3).tolist())))

> **Check.** You should see:
> ```text
> test accuracy: 0.8879
> filled in 0.725, right there 0.9742, others: true class in the top 2 0.9038
> recall per class: {'T-shirt/top': 0.827, 'Trouser': 0.981, 'Pullover': 0.842, 'Dress': 0.893, 'Coat': 0.853, 'Sandal': 0.966, 'Shirt': 0.661, 'Sneaker': 0.95, 'Bag': 0.936, 'Ankle boot': 0.97}
> ```

**What this tells us.** On 10,000 images that no step has touched, the network is a little below its validation accuracy, as expected, because the validation set helped to choose. These are the numbers to report.

## 10. Save the model and the experiment record

Save the weights with `state_dict()`, not the whole object: the file holds only the numbers, and any code with the same `SmallCNN` class can load them. Next to it, write an experiment record: what someone needs to repeat and judge the result.

In [ ]:
# [save]
import json
import platform
import sklearn

torch.save(cnn.state_dict(), "fashion_cnn.pt")
record = {
    "model": "SmallCNN: conv 16 (3x3) - pool - conv 32 (3x3) - pool - linear 64 - linear 10",
    "weights": count(cnn),
    "data": {"name": "Fashion-MNIST (Zalando Research, MIT License)", "sha256": FILES,
             "train": "10,000 of the training file, stratified, random_state=0",
             "valid": "5,000 others of the training file, stratified, random_state=0", "test": "the 10,000 test images"},
    "training": {"optimizer": "Adam", "lr": 0.001, "batch_size": 64, "max_epochs": 30, "patience": 5,
                 "seed": 0, "flip": False, "best_epoch": cnn_best["epoch"], "torch_threads": torch.get_num_threads()},
    "results": {"valid_acc": cnn_best["valid_acc"], "valid_loss": cnn_best["valid_loss"],
                "test_acc": round((test_pred == y_test).float().mean().item(), 4),
                "fill_in_threshold": 0.9, "test_share_filled_in": round(sure.float().mean().item(), 4)},
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
                 "scikit-learn": sklearn.__version__},
    "limits": "28 x 28 grey catalogue photos, one centred product on black; not tested on real seller photos.",
}
Path("experiment.json").write_text(json.dumps(record, indent=2))
print(sorted(p.name for p in Path(".").iterdir() if not p.name.startswith(".")))
print("fashion_cnn.pt:", Path("fashion_cnn.pt").stat().st_size, "bytes")

> **Check.** You should see:
> ```text
> ['data', 'experiment.json', 'fashion_cnn.pt']
> fashion_cnn.pt: 427333 bytes
> ```

Now load the weights into a **new** network, as the shop's app would, and check that it gives the same answers.

In [ ]:
# [reload]
loaded = SmallCNN()
loaded.load_state_dict(torch.load("fashion_cnn.pt", weights_only=True))
loaded.eval()                                   # evaluation mode before any use
same = torch.equal(predict(loaded, x_test).argmax(1), test_pred)
print("same predictions on all 10,000 test images:", same)

> **Check.** You should see:
> ```text
> same predictions on all 10,000 test images: True
> ```

## 11. A prediction on new images

`suggest()` is what Brightloom's upload page would call: one image in, the category to fill in (or two choices) out. Here it gets the first eight test images. The model never trained on them.

In [ ]:
# [suggest]
def suggest(model, image, threshold=0.9):
    """One 28 x 28 image (0..255) -> what the upload page shows, and the top 3 probabilities."""
    x = torch.tensor(np.asarray(image), dtype=torch.float32).reshape(1, 1, 28, 28) / 255
    probs = predict(model, x)[0]
    top = probs.topk(3)
    choices = [(CLASSES[c], round(p, 3)) for p, c in zip(top.values.tolist(), top.indices.tolist())]
    shown = f"fill in: {choices[0][0]}" if choices[0][1] >= threshold else f"ask: {choices[0][0]} or {choices[1][0]}?"
    return shown, choices


for i in range(8):
    shown, choices = suggest(loaded, test_images[i])
    print(f"{i}  label {CLASSES[test_labels[i]]:<11} -> {shown:<28} {choices}")
show_images(test_images[:8], [CLASSES[test_labels[i]] for i in range(8)], columns=8)

> **Check.** You should see:
> ```text
> 0  label Ankle boot  -> fill in: Ankle boot          [('Ankle boot', 0.976), ('Sneaker', 0.016), ('Sandal', 0.007)]
> 1  label Pullover    -> fill in: Pullover            [('Pullover', 0.938), ('Coat', 0.06), ('Shirt', 0.001)]
> 2  label Trouser     -> fill in: Trouser             [('Trouser', 1.0), ('Dress', 0.0), ('Sandal', 0.0)]
> 3  label Trouser     -> fill in: Trouser             [('Trouser', 1.0), ('Dress', 0.0), ('Sandal', 0.0)]
> 4  label Shirt       -> ask: Shirt or Coat?          [('Shirt', 0.867), ('Coat', 0.056), ('T-shirt/top', 0.031)]
> 5  label Trouser     -> fill in: Trouser             [('Trouser', 1.0), ('Dress', 0.0), ('Pullover', 0.0)]
> 6  label Coat        -> fill in: Coat                [('Coat', 0.994), ('Pullover', 0.005), ('Shirt', 0.001)]
> 7  label Shirt       -> ask: Coat or Shirt?          [('Coat', 0.59), ('Shirt', 0.391), ('Pullover', 0.014)]
> ```

**What this tells us.** Where the photo is clear, the model is sure and right. Where it hesitates, it hesitates between the classes of the confusion matrix (shirt, coat, pullover), and the true category is among the two choices.

## Next

You built an image classifier end to end: tensors from raw bytes, a baseline, two networks under one split, early stopping, the confusion matrix and the errors, an honest augmentation test, an operating point, one test, a saved model with its record, and a suggestion for new images. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/classify-clothing-images/) has the recommendations, the limits, responsible use, the portfolio template and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Deep network or gradient boosting?](https://learning.nextia-ai.com/courses/deep-learning/m08/deep-network-or-gradient-boosting/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m08/classify-clothing-images/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The images are real product photos from Fashion-MNIST (Zalando Research, MIT License); the shop is made up.